Do installations

In [ ]:
# ✅ Run once per runtime
!pip -q install geopandas pyogrio shapely rtree folium scikit-learn


Mount the google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Import the parameters, paths, and the imports

In [ ]:
import os, re
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString

# --- Paths ---
CRASHES_PATH   = "/content/drive/My Drive/Crashes_in_DC.csv"
BIKELANES_PATH = "/content/drive/My Drive/Bicycle_Lanes.geojson"
OUT_DIR        = "/content/drive/My Drive/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# --- Analysis window ---
DATE_MIN = "2020-01-01"
DATE_MAX = "2025-04-30"

# --- Optional filters ---
MAR_SCORE_MIN = 100   # set to None to disable

# --- Bike-injury columns (will be created if missing) ---
fatal_cols = ["FATAL_BICYCLIST","FATAL_DRIVER","FATAL_PEDESTRIAN","FATALPASSENGER","FATALOTHER"]
major_cols = ["MAJORINJURIES_BICYCLIST","MAJORINJURIES_DRIVER","MAJORINJURIES_PEDESTRIAN","MAJORINJURIESPASSENGER","MAJORINJURIESOTHER"]
minor_cols = ["MINORINJURIES_BICYCLIST","MINORINJURIES_DRIVER","MINORINJURIES_PEDESTRIAN","MINORINJURIESPASSENGER","MINORINJURIESOTHER"]
bike_cols  = ["MAJORINJURIES_BICYCLIST","MINORINJURIES_BICYCLIST","UNKNOWNINJURIES_BICYCLIST","FATAL_BICYCLIST"]


Load, clean and filter in car accidents

In [ ]:
df = pd.read_csv(CRASHES_PATH, low_memory=False)

# Coordinates → numeric, drop bad
df = df.dropna(subset=["LATITUDE","LONGITUDE"]).copy()
df["LATITUDE"]  = pd.to_numeric(df["LATITUDE"],  errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
df = df.dropna(subset=["LATITUDE","LONGITUDE"]).copy()

# Date window
df["FROMDATE"] = pd.to_datetime(df["FROMDATE"], errors="coerce")
df = df[(df["FROMDATE"] >= DATE_MIN) & (df["FROMDATE"] <= DATE_MAX)].copy()

# Optional MAR_SCORE filter
if "MAR_SCORE" in df.columns and MAR_SCORE_MIN is not None:
    df["MAR_SCORE"] = pd.to_numeric(df["MAR_SCORE"], errors="coerce")
    df = df[df["MAR_SCORE"] >= MAR_SCORE_MIN].copy()

# Ensure injury columns exist & numeric
for cols in (fatal_cols, major_cols, minor_cols, bike_cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# Extract a simple street name from ADDRESS (optional, useful for grouping)
def extract_street_name(addr):
    if pd.isna(addr): return None
    s = str(addr).strip()
    if not s or s.lower() in {"nan","none","null"}: return None
    s = re.sub(r"(#\s*\w+|APT\s*\w+|UNIT\s*\w+)$", "", s, flags=re.IGNORECASE).strip()
    s = re.sub(r"^\d+[A-Z]?(?:-\d+)?\s+", "", s).strip()      # drop house number
    s = re.sub(r"^(BLOCK OF|BLK|BLOCK)\s+", "", s, flags=re.IGNORECASE).strip()
    s = re.sub(r"[,\.;:]+$", "", s).strip()
    return s if s else None

df["street_name"] = df["ADDRESS"].apply(extract_street_name) if "ADDRESS" in df.columns else None

print(f"Crashes after clean/date filters: {len(df):,}")


Only including accidents that involve bike lanes

In [ ]:
# Keep rows where any bicyclist injury/fatal is recorded
df_bike = df[df[bike_cols].sum(axis=1) > 0].copy()
print(f"Bike-involved crashes: {len(df_bike):,}")


Load in the bike lanes as lines

In [ ]:
gdf_lanes_4326 = gpd.read_file(BIKELANES_PATH)

# If layer lacks CRS, assume WGS84
if gdf_lanes_4326.crs is None:
    gdf_lanes_4326.set_crs(epsg=4326, inplace=True, allow_override=True)

# Drop empties / invalids
gdf_lanes_4326 = gdf_lanes_4326[
    gdf_lanes_4326.geometry.notna() & ~gdf_lanes_4326.geometry.is_empty
].copy()

# Keep only line-like geometries
gdf_lanes_4326 = gdf_lanes_4326[gdf_lanes_4326.geometry.geom_type.isin(
    ["LineString","MultiLineString"]
)].copy()

print(f"Bike lane features (valid line-ish): {len(gdf_lanes_4326):,}")


Project to meters

In [ ]:
# Points (crashes) in 4326
gdf_bike_4326 = gpd.GeoDataFrame(
    df_bike,
    geometry=gpd.points_from_xy(df_bike["LONGITUDE"], df_bike["LATITUDE"]),
    crs="EPSG:4326"
)

# Project both to 3857 (meters) for later distance work
gdf_bike_3857  = gdf_bike_4326.to_crs(epsg=3857)
gdf_lanes_3857 = gdf_lanes_4326.to_crs(epsg=3857)

# Keep original lat/lon on the point GDF for tooltips/outputs
gdf_bike_3857["LATITUDE"]  = gdf_bike_4326["LATITUDE"].values
gdf_bike_3857["LONGITUDE"] = gdf_bike_4326["LONGITUDE"].values

print("CRS (points):", gdf_bike_3857.crs)
print("CRS (lanes): ", gdf_lanes_3857.crs)
print(f"Bike points in meters: {len(gdf_bike_3857):,}")
print(f"Lane features in meters: {len(gdf_lanes_3857):,}")


Brute force distance calculations

In [ ]:
# Cell 8 — Pure brute-force nearest bike lane (no prefilters, no classifications)
from shapely.geometry import LineString, MultiLineString
from shapely.ops import nearest_points
import numpy as np
import math

# 1) Flatten lanes to simple LineStrings (keep original index + optional label)
lane_lines = []
lane_orig_index = []
lane_label_col = None
for cand in ["NAME","STREET","FACILITY","FACILITY_T","TYPE","LABEL"]:
    if cand in gdf_lanes_3857.columns:
        lane_label_col = cand
        break
lane_label_vals = []

for idx, geom in zip(gdf_lanes_3857.index, gdf_lanes_3857.geometry):
    if geom is None:
        continue
    if isinstance(geom, LineString):
        lane_lines.append(geom)
        lane_orig_index.append(idx)
        lane_label_vals.append(gdf_lanes_3857.at[idx, lane_label_col] if lane_label_col else None)
    elif isinstance(geom, MultiLineString):
        for sub in geom.geoms:
            lane_lines.append(sub)
            lane_orig_index.append(idx)
            lane_label_vals.append(gdf_lanes_3857.at[idx, lane_label_col] if lane_label_col else None)

print(f"Flattened lanes: {len(lane_lines)} LineStrings (from {len(gdf_lanes_3857)} features)")

# 2) For each crash: compute exact min distance across ALL lines
n = len(gdf_bike_3857)
nearest_lane_index = np.empty(n, dtype=object)
nearest_lane_label = np.empty(n, dtype=object)
nearest_dist_m     = np.empty(n, dtype=float)
nearest_onlane_x   = np.empty(n, dtype=float)
nearest_onlane_y   = np.empty(n, dtype=float)

for i, pt in enumerate(gdf_bike_3857.geometry):
    best_d = math.inf
    best_j = -1
    for j, line in enumerate(lane_lines):
        d = pt.distance(line)  # true point-to-line distance (meters)
        if d < best_d:
            best_d = d
            best_j = j

    nearest_dist_m[i]     = best_d
    nearest_lane_index[i] = lane_orig_index[best_j] if best_j >= 0 else None
    nearest_lane_label[i] = lane_label_vals[best_j] if best_j >= 0 else None

    if best_j >= 0:
        _, q = nearest_points(pt, lane_lines[best_j])
        nearest_onlane_x[i] = q.x
        nearest_onlane_y[i] = q.y
    else:
        nearest_onlane_x[i] = np.nan
        nearest_onlane_y[i] = np.nan

# 3) Assemble results
gdf_with_dist = gdf_bike_3857.copy()
gdf_with_dist["nearest_lane_index"] = nearest_lane_index
gdf_with_dist["nearest_lane_label"] = nearest_lane_label
gdf_with_dist["dist_to_lane_m"]     = nearest_dist_m
gdf_with_dist["nearest_onlane_x"]   = nearest_onlane_x
gdf_with_dist["nearest_onlane_y"]   = nearest_onlane_y

print("\nDistance summary (meters):")
print(gdf_with_dist["dist_to_lane_m"].describe())
print("\nSample:")
print(gdf_with_dist[["dist_to_lane_m","nearest_lane_label"]].head())


Filter out the crashes that are within the threshold

In [ ]:
# ==== TEMPORARY DISTANCE THRESHOLD (easy to adjust) ====
FAR_THRESH_M = 100   # <-- change this anytime (e.g., 50, 150)

# Keep only crashes farther than the threshold from any bike lane
far_df = gdf_with_dist[gdf_with_dist["dist_to_lane_m"] > FAR_THRESH_M].copy()

print(f"Far-from-lane crashes (> {FAR_THRESH_M} m): {len(far_df)} / {len(gdf_with_dist)} total")


Connect each of the car accidents to streets using the DDOT's lines.

Parameters for DDOT

In [ ]:
import re, pandas as pd, geopandas as gpd

# Your roadway layer (DDOT centerlines)
SPEED_GEOJSON_PATH = "/content/drive/My Drive/Roadway_SubBlock.geojson"

# CRS + snap threshold (easy to tweak)
CRS_WGS84     = 4326
CRS_METERS    = 3857
MAX_LATERAL_M = 30   # <-- change this later if needed




Load DDOT lines

In [ ]:
ddot_lines = gpd.read_file(SPEED_GEOJSON_PATH)

# Ensure CRS, drop empties, project to meters
if ddot_lines.crs is None:
    ddot_lines.set_crs(epsg=CRS_WGS84, inplace=True, allow_override=True)
ddot_lines = ddot_lines[ddot_lines.geometry.notna() & ~ddot_lines.geometry.is_empty].copy()
ddot_lines = ddot_lines.to_crs(epsg=CRS_METERS)

# Ensure a 'street_base' string column (derive from a name-like field if present)
name_col = None
for c in ["STNAME","NAME","FULLNAME","ROADNAME","STREET","LABEL"]:
    if c in ddot_lines.columns:
        name_col = c
        break

def mk_street_base(s):
    if pd.isna(s): return ""
    s = str(s).strip().upper()
    s = re.sub(r"^\d+[A-Z]?(?:-\d+)?\s+", "", s)  # drop leading numbers
    s = re.sub(r"[,\.;:]+$", "", s)               # drop trailing punctuation
    s = re.sub(r"\s+", " ", s)                    # collapse spaces
    return s

if "street_base" not in ddot_lines.columns:
    ddot_lines["street_base"] = ddot_lines[name_col].apply(mk_street_base) if name_col else ""

print("DDOT lines ready → features:", len(ddot_lines), "| CRS:", ddot_lines.crs)



Assign the car accidents to the DDOT lines

In [ ]:
# 3) ASSIGN FAR-FROM-LANE BIKE CRASHES — map to nearest DDOT line, inherit street_base
# Expects: far_df (already filtered to > FAR_THRESH_M) with geometry in meters

# Ensure DDOT CRS matches far_df
if ddot_lines.crs != far_df.crs:
    ddot_lines = ddot_lines.to_crs(far_df.crs)

# Only keep needed DDOT fields
ddot_min = ddot_lines[["street_base", "geometry"]].copy()

cr2line = (
    gpd.sjoin_nearest(far_df, ddot_min, how="left", distance_col="cr_dist_to_line_m")
      .rename(columns={"index_right": "ddot_idx"})
      .reset_index()
      .rename(columns={"index": "crash_index"})
)

cr2line["cr_valid_line"] = cr2line["cr_dist_to_line_m"] <= MAX_LATERAL_M
cr2line["street_base"]   = cr2line["street_base"].fillna("").astype(str)

print("Far crashes valid to street:", int(cr2line["cr_valid_line"].sum()), "/", len(cr2line))


Complete Linkage

In [ ]:
# CELL — Complete-linkage clustering (per street), keeping compact hotspots
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import pandas as pd
import geopandas as gpd

# Params (easy to tweak)
CLUSTER_DIST_M = FAR_THRESH_M   # max diameter of a cluster (meters)
MIN_POINTS     = 3    # minimum crashes per cluster to keep

# Work on valid snaps only
far_valid = cr2line[cr2line["cr_valid_line"]].copy()
if far_valid.empty:
    print("No far-from-lane crashes valid to a street. Nothing to cluster.")
    clustered = far_valid.copy()
    clustered["cluster_id"] = -1
else:
    # Calculate a simple severity score if ingredients exist
    fatal_cols = ["FATAL_BICYCLIST","FATAL_DRIVER","FATAL_PEDESTRIAN","FATALPASSENGER","FATALOTHER"]
    major_cols = ["MAJORINJURIES_BICYCLIST","MAJORINJURIES_DRIVER","MAJORINJURIES_PEDESTRIAN","MAJORINJURIESPASSENGER","MAJORINJURIESOTHER"]
    minor_cols = ["MINORINJURIES_BICYCLIST","MINORINJURIES_DRIVER","MINORINJURIES_PEDESTRIAN","MINORINJURIESPASSENGER","MINORINJURIESOTHER"]
    for cols in (fatal_cols, major_cols, minor_cols):
        for c in cols:
            if c not in far_valid.columns:
                far_valid[c] = 0
            far_valid[c] = pd.to_numeric(far_valid[c], errors="coerce").fillna(0)
    far_valid["SEVERITY_SCORE"] = (
          7 * far_valid[fatal_cols].sum(axis=1)
        + 4 * far_valid[major_cols].sum(axis=1)
        + 1 * far_valid[minor_cols].sum(axis=1)
    )
    # row-level totals
    far_valid["TOTAL_FATALITIES"] = far_valid[fatal_cols].sum(axis=1)
    far_valid["TOTAL_MAJOR_INJURIES"] = far_valid[major_cols].sum(axis=1)
    far_valid["TOTAL_MINOR_INJURIES"] = far_valid[minor_cols].sum(axis=1)

    # uncapped severity
    far_valid["SEVERITY_SCORE"] = (
          7 * far_valid["TOTAL_FATALITIES"]
        + 4 * far_valid["TOTAL_MAJOR_INJURIES"]
        + 1 * far_valid["TOTAL_MINOR_INJURIES"]
    )

    # capped severity
    far_valid["CAPPED_SEVERITY"] = np.where(
        far_valid["TOTAL_FATALITIES"] > 0, 7,
        np.where(
            far_valid["TOTAL_MAJOR_INJURIES"] > 0, 4,
            np.where(far_valid["TOTAL_MINOR_INJURIES"] > 0, 1, 0)
        )
    )

    # severe/fatal flags
    far_valid["IS_SEVERE_CRASH"] = (
        (far_valid["TOTAL_FATALITIES"] > 0) |
        (far_valid["TOTAL_MAJOR_INJURIES"] > 0)
    ).astype(int)

    far_valid["IS_FATAL_CRASH"] = (
        far_valid["TOTAL_FATALITIES"] > 0
    ).astype(int)

    parts = []
    global_id = 0
    for street, grp in far_valid.groupby(far_valid["street_base"].fillna("").astype(str)):
        if len(grp) < MIN_POINTS:
            continue
        XY = np.c_[grp.geometry.x.values, grp.geometry.y.values]
        model = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=CLUSTER_DIST_M,
            linkage="complete"
        )
        labels = model.fit_predict(XY)
        grp = grp.copy()
        # keep only real clusters (size >= MIN_POINTS)
        grp["local_label"] = labels
        counts = grp["local_label"].value_counts()
        keep = counts[counts >= MIN_POINTS].index
        grp = grp[grp["local_label"].isin(keep)].copy()
        if grp.empty:
            continue
        # make globally unique cluster ids
        label_map = {lab: (global_id + i) for i, lab in enumerate(sorted(keep))}
        grp["cluster_id"] = grp["local_label"].map(label_map)
        global_id += len(keep)
        parts.append(grp.drop(columns=["local_label"]))

    clustered = gpd.GeoDataFrame(pd.concat(parts, axis=0), crs=far_valid.crs) if parts else gpd.GeoDataFrame(geometry=[], crs=far_valid.crs)

print(f"Clustered points kept: {0 if clustered.empty else len(clustered)}")


Table

In [ ]:
# =========================
# BIKE CLUSTER TABLE
# =========================

print("\nBuilding upgraded cluster summary table...")

fc = clustered.copy()

summaries = []

if fc.empty:
    print("No clustered crashes available.")
    cluster_simple = pd.DataFrame(columns=[
        "RANK",
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "AVG_SEVERITY",
        "AVG_CAPPED_SEVERITY",
        "SEVERE_CRASH_PERCENTAGE",
        "FATAL_CRASH_PERCENTAGE",
        "AVG_LON",
        "AVG_LAT"
    ])
else:
    for cid in sorted(fc["cluster_id"].unique()):
        group = fc[fc["cluster_id"] == cid].copy()

        n_crashes = len(group)

        severity_sum = group["SEVERITY_SCORE"].sum()
        capped_sum   = group["CAPPED_SEVERITY"].sum()

        avg_severity = severity_sum / n_crashes
        avg_capped   = capped_sum / n_crashes

        severe_pct = 100 * group["IS_SEVERE_CRASH"].mean()
        fatal_pct  = 100 * group["IS_FATAL_CRASH"].mean()

        group_ll = group.to_crs(4326).copy()
        avg_lat = group_ll.geometry.y.mean()
        avg_lon = group_ll.geometry.x.mean()

        summaries.append({
            "N_CRASHES": n_crashes,
            "SEVERITY_SUM": severity_sum,
            "CAPPED_SEVERITY_SUM": capped_sum,
            "AVG_SEVERITY": avg_severity,
            "AVG_CAPPED_SEVERITY": avg_capped,
            "SEVERE_CRASH_PERCENTAGE": severe_pct,
            "FATAL_CRASH_PERCENTAGE": fatal_pct,
            "AVG_LON": avg_lon,
            "AVG_LAT": avg_lat
        })

    cluster_simple = pd.DataFrame(summaries)

    cluster_simple = cluster_simple.sort_values(
        ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    cluster_simple.insert(0, "RANK", np.arange(1, len(cluster_simple) + 1))

print(f"Clusters summarized: {len(cluster_simple)}")
display(cluster_simple.head(10))

Refined table for Pulbication

In [ ]:
# =========================
# BIKE RESULTS TABLE
# =========================

results_table = cluster_simple[
    [
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "AVG_SEVERITY",
        "SEVERE_CRASH_PERCENTAGE",
        "FATAL_CRASH_PERCENTAGE",
        "AVG_LON",
        "AVG_LAT"
    ]
].copy()

print("Bike Results Table:")
display(results_table.head(10))

Capped vs Uncapped Severity

In [ ]:
# =========================
# BIKE: CAPPED-SEVERITY COMPARISON
# =========================

from scipy.stats import spearmanr

# --- Capped severity ranked table ---
cluster_df_capped = cluster_simple.copy()

# Preserve original severity rank
cluster_df_capped = cluster_df_capped.rename(columns={"RANK": "ORIGINAL_RANK"})

# Re-sort by capped severity burden first, then capped severity intensity, then frequency
cluster_df_capped = cluster_df_capped.sort_values(
    ["CAPPED_SEVERITY_SUM", "AVG_CAPPED_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Add capped ranking
cluster_df_capped.insert(0, "CAPPED_RANK", np.arange(1, len(cluster_df_capped) + 1))

print("Capped-Severity Ranked Cluster Table:")
display(cluster_df_capped.head(10))

# --- Spearman correlation ---
rho, pval = spearmanr(
    cluster_df_capped["ORIGINAL_RANK"],
    cluster_df_capped["CAPPED_RANK"]
)

print(f"\nSpearman correlation (rho): {rho:.4f}")
print(f"P-value: {pval:.4e}")

# --- Top-10 overlap ---
k10 = min(10, len(cluster_df_capped))

top_original_10 = set(cluster_df_capped.nsmallest(k10, "ORIGINAL_RANK").index)
top_capped_10   = set(cluster_df_capped.nsmallest(k10, "CAPPED_RANK").index)

overlap_10 = len(top_original_10 & top_capped_10)
overlap_10_pct = overlap_10 / k10 * 100

print(f"\nTop-{k10} overlap: {overlap_10}/{k10} ({overlap_10_pct:.1f}%)")

# --- Top-5 overlap ---
k5 = min(5, len(cluster_df_capped))

top_original_5 = set(cluster_df_capped.nsmallest(k5, "ORIGINAL_RANK").index)
top_capped_5   = set(cluster_df_capped.nsmallest(k5, "CAPPED_RANK").index)

overlap_5 = len(top_original_5 & top_capped_5)
overlap_5_pct = overlap_5 / k5 * 100

print(f"Top-{k5} overlap: {overlap_5}/{k5} ({overlap_5_pct:.1f}%)")

Frequency Comparisons

In [ ]:
# =========================
# BIKE: FREQUENCY-RANKED TABLE
# =========================

from scipy.stats import spearmanr

cluster_df_freq = cluster_df_capped.copy()

# rename for clarity
cluster_df_freq = cluster_df_freq.rename(columns={
    "ORIGINAL_RANK": "SEVERITY_RANK",
    "CAPPED_RANK": "CAPPED_SEVERITY_RANK"
})

# sort by number of crashes
cluster_df_freq = cluster_df_freq.sort_values(
    ["N_CRASHES", "SEVERITY_SUM", "AVG_SEVERITY"],
    ascending=[False, False, False]
).reset_index(drop=True)

# add frequency rank
cluster_df_freq.insert(0, "FREQUENCY_RANK", np.arange(1, len(cluster_df_freq)+1))

print("Frequency Ranked Cluster Table:")
display(cluster_df_freq.head(10))

# =========================
# SPEARMAN CORRELATIONS
# =========================

rho_sf, p_sf = spearmanr(
    cluster_df_freq["SEVERITY_RANK"],
    cluster_df_freq["FREQUENCY_RANK"]
)

rho_cf, p_cf = spearmanr(
    cluster_df_freq["CAPPED_SEVERITY_RANK"],
    cluster_df_freq["FREQUENCY_RANK"]
)

print("\nSpearman Correlations")
print(f"Severity vs Frequency: rho = {rho_sf:.4f}, p = {p_sf:.4e}")
print(f"Capped vs Frequency:   rho = {rho_cf:.4f}, p = {p_cf:.4e}")

# =========================
# TOP-K OVERLAYS
# =========================

for k in [5, 10]:
    k_use = min(k, len(cluster_df_freq))

    top_freq = set(cluster_df_freq.nsmallest(k_use, "FREQUENCY_RANK").index)
    top_sev  = set(cluster_df_freq.nsmallest(k_use, "SEVERITY_RANK").index)
    top_cap  = set(cluster_df_freq.nsmallest(k_use, "CAPPED_SEVERITY_RANK").index)

    overlap_sf = len(top_freq & top_sev)
    overlap_cf = len(top_freq & top_cap)

    print(f"\nTop-{k_use} Overlap")
    print(f"Severity vs Frequency: {overlap_sf}/{k_use} ({100*overlap_sf/k_use:.1f}%)")
    print(f"Capped vs Frequency:   {overlap_cf}/{k_use} ({100*overlap_cf/k_use:.1f}%)")

Threshold Saving

In [ ]:
# =========================
# SAVE BIKE TABLE FOR CURRENT THRESHOLD
# =========================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

SAVE_DIR = "/content/drive/MyDrive/bike_threshold_tables"
os.makedirs(SAVE_DIR, exist_ok=True)

# choose best table to save
save_df = cluster_df_freq.copy()   # most complete table

filename = f"bike_{FAR_THRESH_M}.csv".replace(".", "_")

path = os.path.join(SAVE_DIR, filename)

save_df.to_csv(path, index=False)

print("Saved to:", path)
print("Rows:", len(save_df))

Taking threshold tables from google drive

In [ ]:
bike_75  = pd.read_csv("/content/drive/MyDrive/bike_threshold_tables/bike_75_csv")
bike_100 = pd.read_csv("/content/drive/MyDrive/bike_threshold_tables/bike_100_csv")
bike_125 = pd.read_csv("/content/drive/MyDrive/bike_threshold_tables/bike_125_csv")

Comparison for thresholds (robustness check)

In [ ]:
# =========================
# BIKE THRESHOLD ROBUSTNESS COMPARISON
# same methodology as Night
# compares saved cluster_df_freq tables
# =========================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from pyproj import Transformer
from sklearn.neighbors import BallTree

# project lon/lat to meters
tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

def add_projected_xy(df):
    d = df.copy()
    x, y = tf_xy.transform(
        d["AVG_LON"].to_numpy(),
        d["AVG_LAT"].to_numpy()
    )
    d["X_M"] = x
    d["Y_M"] = y
    return d

def compare_thresholds(df_a, df_b, name_a, name_b,
                       rank_col="SEVERITY_RANK",
                       tolerance_m=100):

    a = add_projected_xy(df_a)
    b = add_projected_xy(df_b)

    tree = BallTree(
        b[["X_M", "Y_M"]].to_numpy(),
        metric="euclidean"
    )

    dist_m, ind = tree.query(
        a[["X_M", "Y_M"]].to_numpy(),
        k=1
    )

    a["MATCH_DIST_M"] = dist_m.flatten()
    a["MATCHED_B_IDX"] = ind.flatten()

    matched = a[a["MATCH_DIST_M"] <= tolerance_m].copy()

    match_pct = 100 * len(matched) / len(a)

    print(f"\n{name_a} vs {name_b}")
    print(f"Matched clusters: {len(matched)}/{len(a)}")
    print(f"Match Percentage: {match_pct:.2f}%")

    if len(matched) < 2:
        print("Not enough matched clusters for Spearman.")
        return

    rank_a = matched[rank_col].to_numpy()
    rank_b = b.iloc[matched["MATCHED_B_IDX"]][rank_col].to_numpy()

    rho, p = spearmanr(rank_a, rank_b)

    print(f"Spearman rho: {rho:.4f}")
    print(f"P-value: {p:.4e}")


print("Running Bike Threshold Robustness Comparisons...")

compare_thresholds(bike_75,  bike_100, "75m",  "100m")
compare_thresholds(bike_100, bike_125, "100m", "125m")
compare_thresholds(bike_75,  bike_125, "75m",  "125m")

Cluster centroids for AADT matching

In [ ]:
# =========================
# BIKE AADT STEP 1
# Build one point per cluster centroid
# =========================

import geopandas as gpd
import pandas as pd
import numpy as np
from pyproj import Transformer

# start from the full saved/final ranked table
bike_base = cluster_df_freq.copy()

# preserve a stable key
bike_base = bike_base.reset_index(drop=True)
bike_base["CLUSTER_KEY"] = bike_base.index

# project AVG_LON / AVG_LAT to EPSG:32618
tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

x_m, y_m = tf_xy.transform(
    bike_base["AVG_LON"].to_numpy(),
    bike_base["AVG_LAT"].to_numpy()
)

bike_base["X_M"] = x_m
bike_base["Y_M"] = y_m

bike_cluster_pts = gpd.GeoDataFrame(
    bike_base,
    geometry=gpd.points_from_xy(bike_base["X_M"], bike_base["Y_M"]),
    crs="EPSG:32618"
)

print("Bike cluster points created:", len(bike_cluster_pts))

Load in AADT and compute nearest segment calculations

In [ ]:
# =========================
# BIKE AADT STEP 2
# Load AADT and match nearest segment
# =========================

import os
import zipfile
import geopandas as gpd
import pandas as pd

zip_path = "/content/drive/MyDrive/Traffic_Volume_-_2024 (3).zip"
extract_path = "/content/drive/MyDrive/aadt"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

AADT_PATH = "/content/drive/MyDrive/aadt/Traffic_Volume_-_2024.shp"

aadt = gpd.read_file(AADT_PATH)

if aadt.crs is None:
    raise ValueError("AADT shapefile has no CRS metadata.")

aadt = aadt.to_crs("EPSG:32618").copy()

AADT_COL = "AADT"

aadt_small = aadt[["geometry", AADT_COL, "ROUTEID"]].copy()
aadt_small[AADT_COL] = pd.to_numeric(aadt_small[AADT_COL], errors="coerce")
aadt_small = aadt_small[aadt_small[AADT_COL].notna() & (aadt_small[AADT_COL] > 0)].copy()
aadt_small = aadt_small.reset_index(drop=True)

bike_aadt_raw = gpd.sjoin_nearest(
    bike_cluster_pts,
    aadt_small,
    how="left",
    distance_col="DIST_TO_AADT_M"
).rename(columns={
    AADT_COL: "MATCHED_AADT",
    "ROUTEID": "MATCHED_ROUTEID"
})

print("Rows before join:", len(bike_cluster_pts))
print("Rows after raw join:", len(bike_aadt_raw))

risk calculations

In [ ]:
# =========================
# BIKE AADT STEP 3
# Tie-break, filter AADT match distance, and compute risk
# =========================

MAX_AADT_MATCH_M = 100

bike_aadt = (
    bike_aadt_raw
    .sort_values(
        by=["CLUSTER_KEY", "DIST_TO_AADT_M", "MATCHED_AADT", "MATCHED_ROUTEID"],
        ascending=[True, True, False, True]
    )
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .sort_values("CLUSTER_KEY")
    .reset_index(drop=True)
)

if len(bike_aadt) != len(bike_cluster_pts):
    raise ValueError("Still not one row per cluster after tie-breaking.")

# =========================
# FILTER OUT AADT MATCHES > 100m
# =========================

pre_filter_n = len(bike_aadt)

bike_aadt["AADT_MATCH_VALID"] = (
    bike_aadt["DIST_TO_AADT_M"] <= MAX_AADT_MATCH_M
)

bike_aadt_out_of_range = bike_aadt[
    ~bike_aadt["AADT_MATCH_VALID"]
].copy()

bike_aadt = bike_aadt[
    bike_aadt["AADT_MATCH_VALID"]
].copy().reset_index(drop=True)

print(f"AADT matches kept within {MAX_AADT_MATCH_M}m: {len(bike_aadt)} / {pre_filter_n}")
print(f"AADT matches removed beyond {MAX_AADT_MATCH_M}m: {len(bike_aadt_out_of_range)} / {pre_filter_n}")

if len(bike_aadt_out_of_range) > 0:
    print("\nRemoved out-of-range AADT matches:")
    display(bike_aadt_out_of_range[[
        "SEVERITY_RANK",
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "MATCHED_AADT",
        "MATCHED_ROUTEID",
        "DIST_TO_AADT_M",
        "AVG_LON",
        "AVG_LAT"
    ]].sort_values("DIST_TO_AADT_M", ascending=False))
else:
    print("✅ No bike AADT matches were over the 100m cap.")

# 5-year exposure to match your current 2020 to 2025 structure
bike_aadt["EXPOSURE_5YR"] = bike_aadt["MATCHED_AADT"] * 365 * 5

bike_aadt["RISK"] = bike_aadt["SEVERITY_SUM"] / bike_aadt["EXPOSURE_5YR"]
bike_aadt["CAPPED_RISK"] = bike_aadt["CAPPED_SEVERITY_SUM"] / bike_aadt["EXPOSURE_5YR"]

bike_aadt["RISK_PER_100M"] = bike_aadt["RISK"] * 100_000_000
bike_aadt["CAPPED_RISK_PER_100M"] = bike_aadt["CAPPED_RISK"] * 100_000_000

bike_risk_base = pd.DataFrame(
    bike_aadt.drop(columns=["geometry", "index_right", "X_M", "Y_M"], errors="ignore")
)

print("Bike risk base table created:", len(bike_risk_base))
display(bike_risk_base.head())

Check max distance to AADT

In [ ]:
# =========================
# CHECK MAX BIKE AADT MATCH DISTANCE
# =========================

print("Bike AADT match distance summary:")
print(
    bike_aadt["DIST_TO_AADT_M"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

max_dist = bike_aadt["DIST_TO_AADT_M"].max()
print(f"\nMaximum distance to matched AADT line: {max_dist:.2f} meters")

# Show the worst matches
worst_bike_aadt_matches = bike_aadt.sort_values(
    "DIST_TO_AADT_M",
    ascending=False
).copy()

display(worst_bike_aadt_matches[[
    "SEVERITY_RANK",
    "N_CRASHES",
    "SEVERITY_SUM",
    "MATCHED_AADT",
    "MATCHED_ROUTEID",
    "DIST_TO_AADT_M",
    "AVG_LON",
    "AVG_LAT"
]].head(20))

Severity-risk table

In [ ]:
# =========================
# BIKE AADT STEP 4
# Risk-ranked table
# =========================

table_severity_risk = bike_risk_base.sort_values(
    ["RISK_PER_100M", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

table_severity_risk.insert(0, "RISK_RANK", np.arange(1, len(table_severity_risk) + 1))
table_severity_risk["RANK_CHANGE"] = (
    table_severity_risk["SEVERITY_RANK"] - table_severity_risk["RISK_RANK"]
)

print("Bike severity-risk ranked table:")
display(table_severity_risk[[
    "RISK_RANK",
    "SEVERITY_RANK",
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].head(10))

Refined Table

In [ ]:
# =========================
# BIKE AADT STEP 4B
# Clean display table — separate from full risk table
# =========================

bike_risk_clean_table = table_severity_risk[[
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].copy()

print("Bike clean severity-risk table:")
display(bike_risk_clean_table.head(10))

Capped risk table

In [ ]:
# =========================
# BIKE AADT STEP 5
# Capped-risk ranked table
# =========================

table_capped_risk = bike_risk_base.sort_values(
    ["CAPPED_RISK_PER_100M", "CAPPED_SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

table_capped_risk.insert(0, "CAPPED_RISK_RANK", np.arange(1, len(table_capped_risk) + 1))
table_capped_risk["CAPPED_RANK_CHANGE"] = (
    table_capped_risk["CAPPED_SEVERITY_RANK"] - table_capped_risk["CAPPED_RISK_RANK"]
)

print("Bike capped-risk ranked table:")
display(table_capped_risk[[
    "CAPPED_RISK_RANK",
    "CAPPED_SEVERITY_RANK",
    "CAPPED_RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].head(10))

Risk Comparisons

In [ ]:
# =========================
# BIKE AADT STEP 6
# Risk comparisons
# =========================

from scipy.stats import spearmanr

# ---- Severity vs Risk ----
rho_sr, p_sr = spearmanr(
    table_severity_risk["SEVERITY_RANK"],
    table_severity_risk["RISK_RANK"]
)

# ---- Capped Severity vs Capped Risk ----
rho_cr, p_cr = spearmanr(
    table_capped_risk["CAPPED_SEVERITY_RANK"],
    table_capped_risk["CAPPED_RISK_RANK"]
)

print("Spearman Correlations\n")
print(f"Severity vs Risk:        rho = {rho_sr:.4f}, p = {p_sr:.4e}")
print(f"Capped Severity vs Risk: rho = {rho_cr:.4f}, p = {p_cr:.4e}")

for k in [5, 10]:
    k_use = min(k, len(table_severity_risk))

    top_sev = set(table_severity_risk.nsmallest(k_use, "SEVERITY_RANK").index)
    top_risk = set(table_severity_risk.nsmallest(k_use, "RISK_RANK").index)

    overlap_sr = len(top_sev & top_risk)

    print(f"\nTop-{k_use} Severity vs Risk Overlap: {overlap_sr}/{k_use} ({100*overlap_sr/k_use:.1f}%)")

for k in [5, 10]:
    k_use = min(k, len(table_capped_risk))

    top_cap = set(table_capped_risk.nsmallest(k_use, "CAPPED_SEVERITY_RANK").index)
    top_caprisk = set(table_capped_risk.nsmallest(k_use, "CAPPED_RISK_RANK").index)

    overlap_cr = len(top_cap & top_caprisk)

    print(f"Top-{k_use} Capped Severity vs Capped Risk Overlap: {overlap_cr}/{k_use} ({100*overlap_cr/k_use:.1f}%)")

Bike Severity vs DDOT

In [ ]:
# =========================
# BIKE vs DDOT HIGH INJURY NETWORK
# Severity-ranked comparison
# same style as Night
# =========================

import geopandas as gpd
import pandas as pd
import numpy as np
from pyproj import Transformer

# -------------------------
# rebuild bike severity points
# -------------------------

bike_base = cluster_df_freq.copy().reset_index(drop=True)
bike_base["CLUSTER_KEY"] = bike_base.index

tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

x_m, y_m = tf_xy.transform(
    bike_base["AVG_LON"].to_numpy(),
    bike_base["AVG_LAT"].to_numpy()
)

bike_base["X_M"] = x_m
bike_base["Y_M"] = y_m

bike_pts = gpd.GeoDataFrame(
    bike_base,
    geometry=gpd.points_from_xy(bike_base["X_M"], bike_base["Y_M"]),
    crs="EPSG:32618"
)

# -------------------------
# DDOT prep
# -------------------------

DDOT_PATH = "/content/drive/MyDrive/High_Injury_Network.geojson"
ddot = gpd.read_file(DDOT_PATH).to_crs("EPSG:32618").copy()

ddot["DDOT_PRIORITY"] = np.select(
    [
        ddot["TIER_1"] == 1,
        ddot["TIER_2"] == 1,
        ddot["TIER_3"] == 1
    ],
    [1, 2, 3],
    default=99
)

# -------------------------
# nearest match
# -------------------------

bike_ddot = gpd.sjoin_nearest(
    bike_pts,
    ddot,
    how="left",
    distance_col="DIST_TO_HIN_M"
)

bike_ddot = (
    bike_ddot
    .sort_values(["SEVERITY_RANK", "DIST_TO_HIN_M"])
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .copy()
)

print("Bike severity clusters matched:", len(bike_ddot))

# -------------------------
# compare top-k
# -------------------------

for k in [5, 10]:
    k_use = min(k, len(bike_ddot))

    top_bike = bike_ddot.nsmallest(k_use, "SEVERITY_RANK").copy()
    top_ddot = ddot.nsmallest(k_use, "DDOT_PRIORITY").copy()

    top_ddot_ids = set(top_ddot.index.tolist())

    overlap = len(
        set(top_bike["index_right"].dropna().astype(int)) &
        top_ddot_ids
    )

    prox = int((top_bike["DIST_TO_HIN_M"] <= 100).sum())

    print(f"\nTop-{k_use} Comparison")
    print(f"Tier overlap: {overlap}/{k_use} ({100*overlap/k_use:.1f}%)")
    print(f"Within 100m of any HIN: {prox}/{k_use} ({100*prox/k_use:.1f}%)")

    # --- ADD THIS at the end of the DDOT comparison cell ---

k_use = min(20, len(bike_ddot))

top_bike = bike_ddot.nsmallest(k_use, "SEVERITY_RANK").copy()
top_ddot = ddot.nsmallest(k_use, "DDOT_PRIORITY").copy()

top_ddot_ids = set(top_ddot.index.tolist())

overlap = len(
    set(top_bike["index_right"].dropna().astype(int)) &
    top_ddot_ids
)

prox = int((top_bike["DIST_TO_HIN_M"] <= 100).sum())

print(f"\nTop-{k_use} Comparison")
print(f"Tier overlap: {overlap}/{k_use} ({100*overlap/k_use:.1f}%)")
print(f"Within 100m of any HIN: {prox}/{k_use} ({100*prox/k_use:.1f}%)")

DDOT vs Risk comparison

In [ ]:
# =========================
# BIKE RISK vs DDOT HIGH INJURY NETWORK
# same style as Night
# includes Top-5, Top-10, and Top-20
# =========================

import geopandas as gpd
import numpy as np
from pyproj import Transformer

# -------------------------
# rebuild risk-ranked cluster points
# -------------------------

bike_risk_base = table_severity_risk.copy().reset_index(drop=True)
bike_risk_base["CLUSTER_KEY"] = bike_risk_base.index

tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

x_m, y_m = tf_xy.transform(
    bike_risk_base["AVG_LON"].to_numpy(),
    bike_risk_base["AVG_LAT"].to_numpy()
)

bike_risk_base["X_M"] = x_m
bike_risk_base["Y_M"] = y_m

bike_risk_pts = gpd.GeoDataFrame(
    bike_risk_base,
    geometry=gpd.points_from_xy(bike_risk_base["X_M"], bike_risk_base["Y_M"]),
    crs="EPSG:32618"
)

# -------------------------
# ensure DDOT priority exists
# -------------------------

if "DDOT_PRIORITY" not in ddot.columns:
    ddot = ddot.to_crs("EPSG:32618").copy()
    ddot["DDOT_PRIORITY"] = np.select(
        [
            ddot["TIER_1"] == 1,
            ddot["TIER_2"] == 1,
            ddot["TIER_3"] == 1
        ],
        [1, 2, 3],
        default=99
    )

# -------------------------
# nearest HIN match
# -------------------------

bike_risk_ddot = gpd.sjoin_nearest(
    bike_risk_pts,
    ddot,
    how="left",
    distance_col="DIST_TO_HIN_M"
)

bike_risk_ddot = (
    bike_risk_ddot
    .sort_values(["RISK_RANK", "DIST_TO_HIN_M"])
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .copy()
)

print("Bike risk clusters matched:", len(bike_risk_ddot))

# -------------------------
# compare top-k
# -------------------------

for k in [5, 10, 20]:
    k_use = min(k, len(bike_risk_ddot))

    top_bike = bike_risk_ddot.nsmallest(k_use, "RISK_RANK").copy()
    top_ddot = ddot.nsmallest(k_use, "DDOT_PRIORITY").copy()

    top_ddot_ids = set(top_ddot.index.tolist())

    overlap = len(
        set(top_bike["index_right"].dropna().astype(int)) &
        top_ddot_ids
    )

    prox = int((top_bike["DIST_TO_HIN_M"] <= 100).sum())

    print(f"\nTop-{k_use} Comparison")
    print(f"Tier overlap: {overlap}/{k_use} ({100*overlap/k_use:.1f}%)")
    print(f"Within 100m of any HIN: {prox}/{k_use} ({100*prox/k_use:.1f}%)")

Comput DBSCAN ranks

In [ ]:
# =========================
# BIKE DBSCAN COMPARISON
# baseline 100m version
# =========================

from sklearn.cluster import DBSCAN
import pandas as pd
import numpy as np

# ---------------------------------
# DBSCAN on the same eligible crash set
# as the primary method
# ---------------------------------

eps_m = 100
min_pts = 2

# Use far_valid, since it already has:
# SEVERITY_SCORE, CAPPED_SEVERITY, etc.
db_df = far_valid.copy()

# projected coordinates already live in geometry
XY_db = np.column_stack([
    db_df.geometry.x.to_numpy(),
    db_df.geometry.y.to_numpy()
])

db = DBSCAN(eps=eps_m, min_samples=min_pts, metric="euclidean")
labels = db.fit_predict(XY_db)

db_df["DBSCAN_CLUSTER"] = labels

# remove noise points
db_df = db_df[db_df["DBSCAN_CLUSTER"] != -1].copy()

print("Clusters found:", db_df["DBSCAN_CLUSTER"].nunique())
print("Points kept:", len(db_df))

# ---------------------------------
# Build severity-ranked DBSCAN table
# ---------------------------------

rows = []

for cid in sorted(db_df["DBSCAN_CLUSTER"].unique()):
    g = db_df[db_df["DBSCAN_CLUSTER"] == cid].copy()
    g_ll = g.to_crs(4326).copy()

    rows.append({
        "N_CRASHES": len(g),
        "SEVERITY_SUM": g["SEVERITY_SCORE"].sum(),
        "AVG_SEVERITY": g["SEVERITY_SCORE"].mean(),
        "AVG_LON": g_ll.geometry.x.mean(),
        "AVG_LAT": g_ll.geometry.y.mean()
    })

dbscan_table = pd.DataFrame(rows)

dbscan_table = dbscan_table.sort_values(
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

dbscan_table.insert(0, "DBSCAN_RANK", np.arange(1, len(dbscan_table)+1))

print("\nTop DBSCAN Clusters")
display(dbscan_table.head(10))

DBSCAN vs Severity Sum

In [ ]:
# =========================
# BIKE DBSCAN vs PRIMARY METHOD
# TRUE TOP-K OVERLAY
# =========================

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from pyproj import Transformer

tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

def add_projected_xy(df):
    d = df.copy()
    x, y = tf_xy.transform(
        d["AVG_LON"].to_numpy(),
        d["AVG_LAT"].to_numpy()
    )
    d["X_M"] = x
    d["Y_M"] = y
    return d

def compare_dbscan_to_primary(primary_df, dbscan_df,
                              primary_rank_col="SEVERITY_RANK",
                              db_rank_col="DBSCAN_RANK",
                              tolerance_m=100):
    p = add_projected_xy(primary_df).copy()
    d = add_projected_xy(dbscan_df).copy()

    # nearest DBSCAN cluster for each primary cluster
    tree = BallTree(d[["X_M", "Y_M"]].to_numpy(), metric="euclidean")
    dist_m, ind = tree.query(p[["X_M", "Y_M"]].to_numpy(), k=1)

    p["MATCH_DIST_M"] = dist_m.flatten()
    p["MATCHED_DB_IDX"] = ind.flatten()

    print("\nDBSCAN vs Primary Method")

    for k in [5, 10, 20]:
        k_use = min(k, len(p), len(d))

        # top-k primary clusters
        top_primary = p.nsmallest(k_use, primary_rank_col).copy()

        # among top-k primary, keep only those with a valid DBSCAN match
        top_primary_matched = top_primary[top_primary["MATCH_DIST_M"] <= tolerance_m].copy()

        # DBSCAN clusters matched to top-k primary clusters
        matched_db_indices_from_primary_topk = set(top_primary_matched["MATCHED_DB_IDX"].tolist())

        # actual top-k DBSCAN clusters
        top_dbscan = d.nsmallest(k_use, db_rank_col).copy()
        top_dbscan_indices = set(top_dbscan.index.tolist())

        overlap = len(matched_db_indices_from_primary_topk & top_dbscan_indices)

        print(f"Top-{k_use} Overlap: {overlap}/{k_use} ({100*overlap/k_use:.1f}%)")

compare_dbscan_to_primary(cluster_df_freq, dbscan_table)

DBSCAN vs AADT Normalized complete linkage

In [ ]:
# =====================================================
# BIKE: DBSCAN vs AADT-NORMALIZED COMPLETE LINKAGE
# TRUE TOP-K OVERLAY ONLY
# =====================================================

bike_risk_for_dbscan = table_severity_risk.copy()

compare_dbscan_to_primary(
    primary_df=bike_risk_for_dbscan,
    dbscan_df=dbscan_table,
    primary_rank_col="RISK_RANK",
    db_rank_col="DBSCAN_RANK",
    tolerance_m=100
)

In [ ]:
# CELL — Folium map of far crashes + TOP 10 clusters (fixed RANK + robust)
import folium
from branca.colormap import linear
import numpy as np
import pandas as pd

# --------------------------
# 1) Prep layers to EPSG:4326
# --------------------------
far_points_ll = cr2line[cr2line["cr_valid_line"]].to_crs(4326) if "cr_valid_line" in cr2line.columns else far_df.to_crs(4326)

# Keep only geometry for DDOT lines to avoid Timestamp serialization errors
ddot_ll_min = ddot_lines[["geometry"]].to_crs(4326).copy()

# --------------------------
# 2) Initialize map
# --------------------------
m = folium.Map(location=[38.9072, -77.0369], zoom_start=12, tiles="cartodbpositron")

# DDOT roadways
folium.GeoJson(
    data=ddot_ll_min.__geo_interface__,
    name="DDOT Roadways",
    style_function=lambda _: {"color": "#6c757d", "weight": 1, "opacity": 0.5},
).add_to(m)

# --------------------------
# 3) All far crashes (uniform style)
# --------------------------
fg_far = folium.FeatureGroup(name="Far-from-lane crashes (uniform)", show=True)
for _, r in far_points_ll.iterrows():
    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=3,
        color="#0d6efd",
        fill=True,
        fill_color="#0d6efd",
        fill_opacity=0.7,
        opacity=1.0,
        weight=0
    ).add_to(fg_far)
fg_far.add_to(m)

# --------------------------
# 4) Choose the right cluster summary DF to map
#    Prefer cluster_simple (has RANK), else fall back to cluster_summary_df
# --------------------------
cluster_df = None

if "cluster_simple" in globals() and isinstance(cluster_simple, pd.DataFrame) and not cluster_simple.empty:
    cluster_df = cluster_simple.copy()
elif "cluster_summary_df" in globals() and isinstance(cluster_summary_df, pd.DataFrame) and not cluster_summary_df.empty:
    cluster_df = cluster_summary_df.copy()
    # If this DF doesn't have RANK, create it after sorting
    # Ensure it has the core columns needed
    needed = {"N_CRASHES", "SEVERITY_SUM", "AVG_LON", "AVG_LAT"}
    if not needed.issubset(set(cluster_df.columns)):
        raise ValueError(f"cluster_summary_df is missing required columns: {needed - set(cluster_df.columns)}")

    cluster_df = cluster_df.sort_values(["SEVERITY_SUM", "N_CRASHES"], ascending=[False, False]).reset_index(drop=True)
    cluster_df.insert(0, "RANK", np.arange(1, len(cluster_df) + 1))

# --------------------------
# 5) Plot TOP 10 clusters only (variable size + color)
# --------------------------
if cluster_df is not None and not cluster_df.empty:
    top10 = cluster_df.sort_values(["SEVERITY_SUM", "N_CRASHES"], ascending=[False, False]).head(10).reset_index(drop=True)

    vmin = float(top10["SEVERITY_SUM"].min())
    vmax = float(top10["SEVERITY_SUM"].max())
    if vmin == vmax:
        vmin = 0.0  # avoid divide-by-zero in scaling
    cmap = linear.Reds_09.scale(vmin, vmax)

    fg_cl = folium.FeatureGroup(name="Top 10 clusters (severity)", show=True)

    for _, row in top10.iterrows():
        sev = float(row["SEVERITY_SUM"])
        n_crashes = int(row["N_CRASHES"])
        rank = int(row["RANK"]) if "RANK" in top10.columns else None

        # Radius scaled by severity (bounded)
        if vmax == vmin:
            radius = 10
        else:
            radius = 8 + 22 * (sev - vmin) / (vmax - vmin)

        popup = (
            (f"<b>Rank:</b> {rank}<br>" if rank is not None else "") +
            f"<b>Crashes:</b> {n_crashes}<br>"
            f"<b>Severity sum:</b> {int(sev)}<br>"
            f"<b>Center lon:</b> {float(row['AVG_LON']):.5f}<br>"
            f"<b>Center lat:</b> {float(row['AVG_LAT']):.5f}"
        )

        folium.CircleMarker(
            location=[float(row["AVG_LAT"]), float(row["AVG_LON"])],
            radius=float(radius),
            color=cmap(sev),
            fill=True,
            fill_color=cmap(sev),
            fill_opacity=0.9,
            opacity=1.0,
            weight=2,
            popup=folium.Popup(popup, max_width=360),
            tooltip=(f"Rank {rank} | Sev {int(sev)} | Cr {n_crashes}" if rank is not None else f"Sev {int(sev)} | Cr {n_crashes}")
        ).add_to(fg_cl)

    fg_cl.add_to(m)
    cmap.caption = "Top 10 cluster severity (SEVERITY_SUM)"
    cmap.add_to(m)
else:
    print("No cluster summary available to plot (cluster_simple / cluster_summary_df missing or empty).")

# --------------------------
# 6) Layer control
# --------------------------
folium.LayerControl(collapsed=False).add_to(m)
m
